# TraceletCodeAgent on GAIA

Runs `TraceletCodeAgent` (src/smolagents/tracelet_agent.py) end-to-end via `agent.run()` on the full GAIA validation set (165 questions), using the same tools/dataset/scorer/eval loop as `naiveReAct.ipynb`, for direct comparison against the naive-ReAct baseline.

Unlike `tracelet_sentinelize_check.ipynb` (which calls `_sample_skeleton`/`_execute_candidates` directly, piece by piece), this drives the agent through its normal multi-step loop, so tool registration (`send_tools`) happens automatically inside `agent.run()` -- no manual setup needed.

In [1]:
import os
import sys
sys.path.insert(0, "../examples/open_deep_research")

from dotenv import load_dotenv
load_dotenv()

from smolagents import OpenAIModel

model_name = "gpt-4o"
model = OpenAIModel(model_id=model_name, api_key=os.environ["OPENAI_API_KEY"])

In [2]:
# Reuse the standard GAIA tool stack, same as naiveReAct.ipynb
from common_setup import build_tools

tools, ti_tool, visualizer = build_tools(model)

/Users/poorvag/Work/smolagents/.venv/lib/python3.14/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [3]:
from smolagents.monitoring import LogLevel
from smolagents.tracelet_agent import TraceletCodeAgent

agent = TraceletCodeAgent(
    tools=tools,
    model=model,
    max_steps=50,
    verbosity_level=LogLevel.INFO,
    additional_authorized_imports=["pandas", "numpy", "PIL", "json", "io", "zipfile", "csv", "openpyxl"],
    n_samples=3,
    skeleton_strategy="post_process",  # AST-based, no reliance on model instruction-following
)

In [4]:
# Load GAIA validation set from HuggingFace
import pandas as pd
from common_setup import load_gaia_dataset

SET_TO_RUN = "validation"
eval_ds = load_gaia_dataset(set_to_run=SET_TO_RUN)

print(f"Loaded {len(eval_ds)} examples")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loaded 165 examples


In [ ]:
from common_setup import evaluate_agent, question_scorer

results = evaluate_agent(
    agent,
    eval_ds,
    ti_tool,
    visualizer,
    n_samples=None,
    output_file=f"tracelet_react_{model_name}.jsonl",
    pickle_dir=f"tracelet_react_{model_name}",
)

In [6]:
df = pd.DataFrame(results)
total = len(df)
correct = df["is_correct"].sum()

print("=== TraceletCodeAgent GAIA Evaluation Results ===")
print(f"Overall accuracy:   {correct}/{total} = {correct/total:.1%}")
print(f"Avg time per question: {df['time_taken_seconds'].mean():.1f}s")
print(f"Avg steps per question: {df['num_steps'].mean():.1f}")

total_tokens = df["token_counts"].apply(lambda x: x.get("total_tokens", 0)).mean()
print(f"Avg total tokens per question: {total_tokens:,.0f}")

print(f"\nTool usage (total calls across all questions):")
tool_usage_df = pd.DataFrame(df["tool_usage"].tolist()).sum().sort_values(ascending=False)
for tool, count in tool_usage_df.items():
    if count > 0:
        print(f"  {tool}: {int(count)}")

=== TraceletCodeAgent GAIA Evaluation Results ===
Overall accuracy:   5/20 = 25.0%
Avg time per question: 105.9s
Avg steps per question: 12.6
Avg total tokens per question: 202,987

Tool usage (total calls across all questions):
  web_search: 44
  visit_page: 29
  final_answer: 18
  find_on_page_ctrl_f: 12
  inspect_file_as_text: 5
  page_down: 3
